In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, Subset, DataLoader
import numpy as np

from dataset import create_dataloaders
from cnn_baseline_1d_graph_simple import CNNBaseline1D
from train import train_one_epoch
from evaluate import evaluate, evaluate_top_models_person_cv
from grid_search import run_grid_search, run_person_grid_search

In [2]:
BATCH_SIZE = 8

INTRA_DATA_DIR = "preprocessed_data/Intra"
intra_train_loader, intra_test_loader = create_dataloaders(INTRA_DATA_DIR, BATCH_SIZE, add_person_id=True)

CROSS_DATA_DIR = "preprocessed_data/Cross"
#EPOCHS = 100
#LEARNING_RATE = 5e-3
NUM_CLASSES = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

cross_train_loader, cross_test_loader = create_dataloaders(CROSS_DATA_DIR, BATCH_SIZE, add_person_id=True)

Loading test data...
Loading train data...
Loaded 32 training samples
Loaded 8 test samples
Class distribution in training: [8 8 8 8]
Class distribution in test: [2 2 2 2]
Train batches: 4
Test batches: 1
Using device: cuda
Loading test1 data...
Loading test2 data...
Loading test3 data...
Loading train data...
Loaded 64 training samples
Loaded 48 test samples
Class distribution in training: [16 16 16 16]
Class distribution in test: [12 12 12 12]
Train batches: 8
Test batches: 6


In [3]:
cross_dataset = cross_train_loader.dataset
intra_dataset = intra_train_loader.dataset

def build_person_subsets(dataset, source_name):
    """Build subsets of the dataset grouped by person_id."""
    if dataset.person_ids is None:
        raise ValueError(f"{source_name} dataset must be created with add_person_id=True")

    grouped_indices = {}
    for index, person_id in enumerate(dataset.person_ids):
        grouped_indices.setdefault(int(person_id), []).append(index)

    return [
        (f"{source_name}_{person_id}", Subset(dataset, indices), person_id)
        for person_id, indices in sorted(grouped_indices.items())
    ]


cross_person_splits = build_person_subsets(cross_dataset, "cross")
intra_person_splits = build_person_subsets(intra_dataset, "intra")
person_splits = cross_person_splits + intra_person_splits

print("Person-level folds:")
for split_name, subset, person_id in person_splits:
    print(f"  {split_name}: person_id={person_id}, samples={len(subset)}")

Person-level folds:
  cross_113922: person_id=113922, samples=32
  cross_164636: person_id=164636, samples=32
  intra_105923: person_id=105923, samples=32


In [4]:
cross_person_splits = build_person_subsets(cross_dataset, "cross")
intra_person_splits = build_person_subsets(intra_dataset, "intra")
person_splits = cross_person_splits + intra_person_splits

print("Person-level folds:")
for split_name, subset, person_id in person_splits:
    print(f"  {split_name}: person_id={person_id}, samples={len(subset)}")

Person-level folds:
  cross_113922: person_id=113922, samples=32
  cross_164636: person_id=164636, samples=32
  intra_105923: person_id=105923, samples=32


In [ ]:
param_grid = {
    "learning_rate": [1e-3, 3e-3],
    "hidden_channels": [32, 64],
    "kernel_size": [5],
    "dropout": [0.1, 0.2],
    "weight_decay": [1e-4],
    "batch_size": [8],
}


top_models = run_person_grid_search(
    model_class=CNNBaseline1D,
    param_grid=param_grid,
    person_splits=person_splits,
    num_classes=4,
    epochs=100,
    patience=20,
)

Total configs: 4

Testing parameters:
{'learning_rate': 0.001, 'hidden_channels': 32, 'kernel_size': 5, 'dropout': 0.1, 'weight_decay': 0.0001, 'batch_size': 8}

Fold 1/3 | Test person: 113922
Fold 1 | Epoch 1/100 | Train Acc: 26.56% | Val Acc: 31.25% | Val Loss: 1.3928
Fold 1 | Epoch 2/100 | Train Acc: 34.38% | Val Acc: 15.62% | Val Loss: 1.4403
Fold 1 | Epoch 3/100 | Train Acc: 39.06% | Val Acc: 3.12% | Val Loss: 1.4915
Fold 1 | Epoch 4/100 | Train Acc: 50.00% | Val Acc: 12.50% | Val Loss: 1.4780
Fold 1 | Epoch 5/100 | Train Acc: 39.06% | Val Acc: 25.00% | Val Loss: 1.4619
Fold 1 | Epoch 6/100 | Train Acc: 57.81% | Val Acc: 25.00% | Val Loss: 1.4145
Fold 1 | Epoch 7/100 | Train Acc: 53.12% | Val Acc: 25.00% | Val Loss: 1.4162
Fold 1 | Epoch 8/100 | Train Acc: 56.25% | Val Acc: 28.12% | Val Loss: 1.4170
Fold 1 | Epoch 9/100 | Train Acc: 64.06% | Val Acc: 25.00% | Val Loss: 1.4132
Fold 1 | Epoch 10/100 | Train Acc: 56.25% | Val Acc: 25.00% | Val Loss: 1.3894
Fold 1 | Epoch 11/100 | Tra

In [7]:
nr_of_runs = 25
evaluate_top_models_person_cv(CNNBaseline1D, top_models, person_splits, num_classes=4, device=DEVICE, n_runs=nr_of_runs)



MODEL 1
{'learning_rate': 0.005, 'hidden_channels': 64, 'kernel_size': 5, 'dropout': 0.1, 'weight_decay': 0.0001, 'batch_size': 8}

Run 1/25


TypeError: CNNBaseline1D.__init__() got an unexpected keyword argument 'weight_decay'